# UNIVERSIDAD EXTERNADO DE COLOMBIA · CIENCIA DE DATOS · 2026

## Clasificación multiclase de la estancia hospitalaria

**Predicción de la Estancia Hospitalaria con Aprendizaje Automático — SPARCS CY2021**

Autores: Daniel Vallejo Vecil · Juliana M.

---

Este cuaderno reformula la predicción de la DEH como un problema de clasificación ordinal en cuatro
tramos, en lugar de regresión sobre días. La motivación es doble: la decisión operativa de gestión de
camas no requiere el número exacto de días sino el tramo de permanencia esperado, y la discretización
evita que la cola extrema de la distribución domine la función de pérdida.

## 1. Definición de los tramos

### 1.1 Qué establecen la normativa y la literatura

No existe en la medicina estadounidense una taxonomía oficial de la estancia hospitalaria en varias
clases. Lo que existe son tres anclajes de distinto origen, que aquí se combinan:

**Criterio regulatorio (CMS, regla de las dos medianoches).** Desde octubre de 2013, la admisión
como paciente internado es apropiada cuando se espera que la permanencia cruce dos medianoches; por
debajo de ese referente corresponde estatus de observación. Bajo la fórmula de SPARCS
—`LOS = (fecha egreso − fecha ingreso) + 1`— cruzar dos medianoches equivale a `LOS >= 3`. El primer
corte, por tanto, se sitúa en 2 días.

**Convención de la literatura.** Los estudios de estancia prolongada emplean mayoritariamente un
umbral de 7 días, o bien el percentil 75 de la distribución observada. En este conjunto ambos
criterios son cercanos: el percentil 75 se sitúa en 6 días y el 24,12 % de los episodios supera los 7.

**Criterio del estado de Nueva York.** El NYSDOH no define la estancia prolongada en días absolutos,
sino en relación con la estancia promedio del APR-DRG correspondiente, valor que publica por grupo y
que utiliza para los pagos por transferencia y por caso atípico. La Sección 7 de este cuaderno evalúa
esa definición relativa como esquema alternativo.

### 1.2 Esquema adoptado

| Clase | Tramo | Fundamento | Lectura operativa |
|---|---|---|---|
| 0 | 1–2 días | Por debajo del criterio de las dos medianoches | Admisión de estancia mínima; candidata a revisión de utilización |
| 1 | 3–6 días | Entre el referente normativo y el percentil 75 | Estancia estándar; rotación de cama predecible |
| 2 | 7–13 días | Supera el umbral convencional de estancia prolongada | Ocupación sostenida; requiere planificación de egreso |
| 3 | ≥ 14 días | Cola extrema, en torno al percentil 90 en adelante | Consumo desproporcionado de camas; incluye los casos censurados |

Los tres cortes —2, 6 y 13 días— no son arbitrarios: el primero es normativo, el segundo empírico
(percentil 75) y el tercero clínico-administrativo (dos semanas, próximo al percentil 90). Declararlos
con su origen es lo que distingue una discretización fundamentada de una partición por conveniencia.

## 2. Configuración y carga

In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    cohen_kappa_score, mean_absolute_error, r2_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent
GOLD_PATH = PROJECT_ROOT / "data" / "3_gold"
MODELOS_PATH = PROJECT_ROOT / "modelos"
TABLAS_PATH = PROJECT_ROOT / "tablas"
MODELOS_PATH.mkdir(parents=True, exist_ok=True)
TABLAS_PATH.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NUM = "06"


def guardar_tabla(tabla, nombre):
    if isinstance(tabla, pd.Series):
        tabla = tabla.to_frame()
    archivo = TABLAS_PATH / f"{NOTEBOOK_NUM}_{nombre}.csv"
    tabla.to_csv(archivo, encoding="utf-8-sig")
    print(f"Tabla guardada: {archivo.resolve()}")


# El split viene fijado desde 03_FeatureEngineering. NO se vuelve a particionar.
X_train = pd.read_parquet(GOLD_PATH / "X_train_sparcs.parquet")
X_test = pd.read_parquet(GOLD_PATH / "X_test_sparcs.parquet")
y_train = pd.read_parquet(GOLD_PATH / "y_train_sparcs.parquet")["Length of Stay"]
y_test = pd.read_parquet(GOLD_PATH / "y_test_sparcs.parquet")["Length of Stay"]

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")

X_train: (1672271, 189)   y_train: (1672271,)
X_test:  (418068, 189)   y_test:  (418068,)


## 3. Construcción de la variable multiclase

In [5]:
CORTES = [0, 2, 6, 13, np.inf]
ETIQUETAS = ["0: 1-2 días", "1: 3-6 días", "2: 7-13 días", "3: 14+ días"]


def a_clase(y):
    return pd.cut(y, bins=CORTES, labels=[0, 1, 2, 3]).astype(int)


y_train_clase = a_clase(y_train)
y_test_clase = a_clase(y_test)

distribucion = pd.DataFrame({
    "n_train": y_train_clase.value_counts().sort_index(),
    "pct_train": y_train_clase.value_counts(normalize=True).sort_index() * 100,
    "n_test": y_test_clase.value_counts().sort_index(),
    "pct_test": y_test_clase.value_counts(normalize=True).sort_index() * 100,
    "dias_medios": y_train.groupby(y_train_clase).mean(),
    "dias_mediana": y_train.groupby(y_train_clase).median(),
})
distribucion.index = ETIQUETAS
display(distribucion.round(2))
guardar_tabla(distribucion, "DistribucionClases")

print()
print("Comparabilidad train/test: diferencia máxima de proporción = "
      f"{(distribucion['pct_train'] - distribucion['pct_test']).abs().max():.3f} puntos porcentuales")

,n_train,pct_train,n_test,pct_test,dias_medios,dias_mediana
0: 1-2 días,660194,39.48,165259,39.53,1.56,2.0
1: 3-6 días,608334,36.38,152033,36.37,4.06,4.0
2: 7-13 días,257493,15.40,64491,15.43,9.16,9.0
3: 14+ días,146250,8.75,36285,8.68,25.63,20.0


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_DistribucionClases.csv

Comparabilidad train/test: diferencia máxima de proporción = 0.066 puntos porcentuales


## 4. Entrenamiento de los clasificadores

Se emplean las mismas tres familias del cuaderno 04, con hiperparámetros equivalentes entre modelos
para que las diferencias sean atribuibles al algoritmo. Se guarda `predict_proba` porque se utiliza
en la Sección 6 para derivar una estimación en días.

In [7]:
PARAMS_RF = dict(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
PARAMS_XGB = dict(
    n_estimators=400, max_depth=8, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, tree_method="hist", random_state=42, n_jobs=-1,
)

clasificadores = {
    "Regresión Logística": LogisticRegression(max_iter=1000, n_jobs=-1),
    "Random Forest": RandomForestClassifier(**PARAMS_RF),
    "XGBoost": XGBClassifier(objective="multi:softprob", num_class=4,
                             eval_metric="mlogloss", **PARAMS_XGB),
}

modelos = {}
probabilidades = {}
resultados = []

for nombre, modelo in clasificadores.items():
    print(f"Entrenando: {nombre} ...")
    modelo.fit(X_train, y_train_clase)
    pred = modelo.predict(X_test)
    proba = modelo.predict_proba(X_test)

    fila = {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test_clase, pred),
        "f1_macro": f1_score(y_test_clase, pred, average="macro"),
        "f1_weighted": f1_score(y_test_clase, pred, average="weighted"),
        # Kappa cuadrático: métrica ordinal, penaliza más el error entre clases distantes
        "kappa_cuadratico": cohen_kappa_score(y_test_clase, pred, weights="quadratic"),
        "MAE_ordinal": mean_absolute_error(y_test_clase, pred),
    }
    resultados.append(fila)
    modelos[nombre] = modelo
    probabilidades[nombre] = proba

    print(f"  accuracy={fila['accuracy']:.3f}  F1-macro={fila['f1_macro']:.3f}  "
          f"kappa_cuad={fila['kappa_cuadratico']:.3f}  MAE_ordinal={fila['MAE_ordinal']:.3f}")

    joblib.dump(modelo, MODELOS_PATH / f"06_multiclase_{nombre.replace(' ', '_')}.joblib")

resultados_df = pd.DataFrame(resultados)
display(resultados_df.round(4))
guardar_tabla(resultados_df, "ResultadosMulticlase")

Entrenando: Regresión Logística ...
  accuracy=0.554  F1-macro=0.480  kappa_cuad=0.584  MAE_ordinal=0.516
Entrenando: Random Forest ...
  accuracy=0.581  F1-macro=0.510  kappa_cuad=0.621  MAE_ordinal=0.478
Entrenando: XGBoost ...
  accuracy=0.592  F1-macro=0.528  kappa_cuad=0.643  MAE_ordinal=0.465


,modelo,accuracy,f1_macro,f1_weighted,kappa_cuadratico,MAE_ordinal
0,Regresión Logística,0.5538,0.4801,0.5417,0.5837,0.5163
1,Random Forest,0.5810,0.5104,0.5699,0.6215,0.4784
2,XGBoost,0.5918,0.5284,0.5838,0.6429,0.4645


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_ResultadosMulticlase.csv


### Sobre las métricas empleadas

La exactitud global es engañosa con clases desbalanceadas: un modelo que asigne siempre la clase
mayoritaria ya obtendría una cifra considerable. Por eso se reportan cuatro medidas complementarias:

- **F1-macro** promedia el desempeño de las cuatro clases sin ponderar por tamaño, de modo que el
  fracaso en la clase 3 —la minoritaria y la operativamente más relevante— no queda diluido.
- **Kappa cuadrático** reconoce que las clases están **ordenadas**: confundir la clase 0 con la 3 es
  un error mucho más grave que confundirla con la 1, y esta métrica lo penaliza cuadráticamente.
  Es la medida principal para un problema ordinal.
- **MAE ordinal** expresa el error medio en número de clases de distancia, e indica de forma directa
  cuán lejos queda la predicción del tramo correcto.

La línea base contra la cual interpretar la exactitud es la proporción de la clase mayoritaria, que
figura en la tabla de la Sección 3.

## 5. Matrices de confusión y desempeño por clase

In [8]:
for nombre, modelo in modelos.items():
    pred = modelo.predict(X_test)

    cm = confusion_matrix(y_test_clase, pred)
    cm_df = pd.DataFrame(cm,
                         index=[f"Real {e}" for e in ETIQUETAS],
                         columns=[f"Pred {e}" for e in ETIQUETAS])
    print(f"=== {nombre} ===")
    display(cm_df)
    guardar_tabla(cm_df, f"MatrizConfusionMulticlase_{nombre.replace(' ', '_')}")

    reporte = pd.DataFrame(
        classification_report(y_test_clase, pred, target_names=ETIQUETAS,
                              output_dict=True, zero_division=0)
    ).T
    display(reporte.round(3))
    guardar_tabla(reporte, f"ReportePorClase_{nombre.replace(' ', '_')}")
    print()

=== Regresión Logística ===


,Pred 0: 1-2 días,Pred 1: 3-6 días,Pred 2: 7-13 días,Pred 3: 14+ días
Real 0: 1-2 días,118105,42622,3083,1449
Real 1: 3-6 días,53343,86342,9544,2804
Real 2: 7-13 días,5980,38054,14066,6391
Real 3: 14+ días,1218,12094,9962,13011


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_MatrizConfusionMulticlase_Regresión_Logística.csv


,precision,recall,f1-score,support
0: 1-2 días,0.661,0.715,0.687,165259.000
1: 3-6 días,0.482,0.568,0.521,152033.000
2: 7-13 días,0.384,0.218,0.278,64491.000
3: 14+ días,0.550,0.359,0.434,36285.000
accuracy,0.554,0.554,0.554,0.554
macro avg,0.519,0.465,0.480,418068.000
weighted avg,0.544,0.554,0.542,418068.000


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_ReportePorClase_Regresión_Logística.csv

=== Random Forest ===


...

Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_ReportePorClase_Random_Forest.csv

=== XGBoost ===


,Pred 0: 1-2 días,Pred 1: 3-6 días,Pred 2: 7-13 días,Pred 3: 14+ días
Real 0: 1-2 días,118733,42378,2726,1422
Real 1: 3-6 días,43063,95293,10534,3143
Real 2: 7-13 días,4226,35459,16949,7857
Real 3: 14+ días,826,8954,10062,16443


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_MatrizConfusionMulticlase_XGBoost.csv


,precision,recall,f1-score,support
0: 1-2 días,0.712,0.718,0.715,165259.000
1: 3-6 días,0.523,0.627,0.570,152033.000
2: 7-13 días,0.421,0.263,0.324,64491.000
3: 14+ días,0.570,0.453,0.505,36285.000
accuracy,0.592,0.592,0.592,0.592
macro avg,0.556,0.515,0.528,418068.000
weighted avg,0.586,0.592,0.584,418068.000


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_ReportePorClase_XGBoost.csv



La lectura relevante de la matriz de confusión no es la diagonal sino **dónde caen los errores**. Un
modelo cuyos errores se concentran en clases adyacentes es operativamente útil; uno que confunde la
clase 0 con la 3 no lo es, aunque su exactitud global sea similar.

## 6. Puente con la regresión: estimación en días a partir de las probabilidades

Para comparar este enfoque con los cuadernos de regresión hace falta una predicción en días. Se
obtiene como la esperanza sobre las cuatro clases, usando la mediana en días de cada tramo estimada
en entrenamiento como valor representativo:

$$\hat{y}_i = \sum_{k=0}^{3} P(\text{clase}=k \mid x_i)\; \cdot \; \text{mediana}_k$$

Esta construcción aprovecha la probabilidad completa y no solo la clase más probable, lo que evita
el escalón artificial de asignar un valor constante por tramo.

In [9]:
medianas_clase = y_train.groupby(y_train_clase).median().to_numpy()
print("Mediana de días por clase (entrenamiento):", medianas_clase)

puente = []
for nombre, proba in probabilidades.items():
    pred_dias = proba @ medianas_clase
    puente.append({
        "modelo": f"Multiclase → días ({nombre})",
        "MAE": mean_absolute_error(y_test, pred_dias),
        "RMSE": float(np.sqrt(((y_test - pred_dias) ** 2).mean())),
        "R2": r2_score(y_test, pred_dias),
    })

puente_df = pd.DataFrame(puente)
display(puente_df.round(4))
guardar_tabla(puente_df, "PuenteMulticlaseADias")

print()
print("Compárese con la tabla 04_ComparacionFinalHurdleVsUnico y con 05_ComparacionTransformaciones.")

Mediana de días por clase (entrenamiento): [ 2.  4.  9. 20.]


,modelo,MAE,RMSE,R2
0,Multiclase → días (Regresión Logística),3.1951,6.9225,0.3047
1,Multiclase → días (Random Forest),3.1457,6.9390,0.3014
2,Multiclase → días (XGBoost),3.0282,6.7580,0.3374


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_PuenteMulticlaseADias.csv

Compárese con la tabla 04_ComparacionFinalHurdleVsUnico y con 05_ComparacionTransformaciones.


## 7. Esquema alternativo: tramos relativos al APR-DRG

El criterio vigente en Nueva York define la estancia prolongada en relación con la esperada para el
grupo diagnóstico, no en días absolutos. Esa referencia ya está disponible en el conjunto: la columna
`APR DRG_TE` contiene la media de estancia de cada APR-DRG, calculada sobre el conjunto de
entrenamiento durante el *Target Encoding*, por lo que su uso no introduce fuga de información.

Se construyen cuatro tramos según la razón entre la estancia observada y la esperada del grupo.

In [10]:
COL_ESPERADA = "APR DRG_TE"

if COL_ESPERADA not in X_train.columns:
    print(f"No se encontró la columna '{COL_ESPERADA}'. Revisar el nombre en la capa Gold.")
else:
    razon_train = y_train.to_numpy() / X_train[COL_ESPERADA].to_numpy()
    razon_test = y_test.to_numpy() / X_test[COL_ESPERADA].to_numpy()

    CORTES_REL = [0, 0.75, 1.25, 2.0, np.inf]
    ETIQUETAS_REL = ["0: por debajo de lo esperado", "1: conforme a lo esperado",
                     "2: por encima", "3: atípica (> 2x)"]

    y_train_rel = pd.cut(razon_train, bins=CORTES_REL, labels=[0, 1, 2, 3]).astype(int)
    y_test_rel = pd.cut(razon_test, bins=CORTES_REL, labels=[0, 1, 2, 3]).astype(int)

    dist_rel = pd.DataFrame({
        "n_train": pd.Series(y_train_rel).value_counts().sort_index(),
        "pct_train": pd.Series(y_train_rel).value_counts(normalize=True).sort_index() * 100,
        "dias_medianos": pd.Series(y_train.to_numpy()).groupby(y_train_rel).median(),
    })
    dist_rel.index = ETIQUETAS_REL
    display(dist_rel.round(2))
    guardar_tabla(dist_rel, "DistribucionClasesRelativas")

    modelo_rel = XGBClassifier(objective="multi:softprob", num_class=4,
                               eval_metric="mlogloss", **PARAMS_XGB)
    print("Entrenando XGBoost sobre tramos relativos al APR-DRG ...")
    modelo_rel.fit(X_train, y_train_rel)
    pred_rel = modelo_rel.predict(X_test)

    fila_rel = pd.DataFrame([{
        "esquema": "Relativo al APR-DRG",
        "accuracy": accuracy_score(y_test_rel, pred_rel),
        "f1_macro": f1_score(y_test_rel, pred_rel, average="macro"),
        "kappa_cuadratico": cohen_kappa_score(y_test_rel, pred_rel, weights="quadratic"),
        "MAE_ordinal": mean_absolute_error(y_test_rel, pred_rel),
    }])
    display(fila_rel.round(4))
    guardar_tabla(fila_rel, "ResultadosClasesRelativas")

    joblib.dump(modelo_rel, MODELOS_PATH / "06_multiclase_relativo_aprdrg.joblib")

,n_train,pct_train,dias_medianos
0: por debajo de lo esperado,799282,47.8,2.0
1: conforme a lo esperado,479926,28.7,4.0
2: por encima,247570,14.8,7.0
3: atípica (> 2x),145493,8.7,16.0


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_DistribucionClasesRelativas.csv
Entrenando XGBoost sobre tramos relativos al APR-DRG ...


,esquema,accuracy,f1_macro,kappa_cuadratico,MAE_ordinal
0,Relativo al APR-DRG,0.5722,0.4254,0.3617,0.63


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\06_ResultadosClasesRelativas.csv


Este esquema responde a una pregunta distinta y más interesante desde la gestión: no *cuánto durará
la estancia*, sino *si durará más de lo que su grupo diagnóstico haría esperar*. Una estancia de
veinte días tras un trasplante es normal; una de ocho por una amigdalitis no lo es. El modelo
absoluto no puede distinguir esos dos casos y este sí.

**Advertencia metodológica.** La estancia esperada procede del *Target Encoding*, calculado sobre el
conjunto de entrenamiento. Como la etiqueta se construye a partir de esa misma cantidad, conviene
declarar explícitamente en el documento que la referencia no utiliza información del conjunto de
prueba. Una alternativa más limpia, si el tiempo lo permite, es emplear las estancias promedio por
APR-DRG publicadas por el NYSDOH, que son externas al conjunto de datos.

## 8. Conclusiones del cuaderno

Elementos a documentar en el Capítulo 4:

1. Si la formulación multiclase supera al enfoque de regresión una vez traducida a días (Sección 6),
   y bajo qué métrica.
2. El patrón de la matriz de confusión: si los errores se concentran en clases adyacentes o si el
   modelo confunde tramos distantes.
3. El desempeño en la clase 3, que es la minoritaria y la operativamente decisiva. Un F1-macro muy
   inferior al F1 ponderado indica que el modelo está sacrificando esa clase.
4. La comparación entre el esquema absoluto y el relativo al APR-DRG, y qué implica para la
   definición operativa de estancia prolongada.

**Tablas exportadas:** `06_DistribucionClases`, `06_ResultadosMulticlase`,
`06_MatrizConfusionMulticlase_*`, `06_ReportePorClase_*`, `06_PuenteMulticlaseADias`,
`06_DistribucionClasesRelativas`, `06_ResultadosClasesRelativas`.